# Valuación Institucional Canónica de Aluar Aluminio Argentino S.A.I.C. (ALUA.BA)
**Notebook Maestro Consolidado Autocontenido · Módulos M1 a M13**  
*Cátedra de Economía y Técnica Bursátil — FCE UNCuyo*  
*Analista:* Federico Agustín Chillón

---
### Descripción
Este notebook contiene el desarrollo **100% autocontenido celda a celda** de los 13 módulos del Modelo de Valuación Institucional Canónico de Aluar S.A.I.C. No requiere librerías externas complejas ni archivos faltantes: ejecuta ingesta, estadística descriptiva, análisis DuPont, proyecciones 2026E-2030E, pipeline de Beta, WACC (7.06%), DCF (ARS 1,236.00), Monte Carlo, VaR/CVaR, Cópulas, Criterio de Kelly, Múltiples Comparables y renderizado de figuras.


## Módulo 1: Ingesta de Datos y Construcción del Panel de Mercado

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Parámetros macroeconómicos y de mercado de la valuación oficial
PARAMETROS_OFICIALES = {
    "spot_ars": 982.50,
    "ccl_ars": 1584.25,
    "rf_us": 0.0470,          # Tasa Libre de Riesgo 10Y (4.70%)
    "embi_pb": 441.0,         # Riesgo País EMBI+ AR (441 pb)
    "embi_pct": 0.0441,       # 4.41%
    "lambda_aluar": 0.20,     # Coeficiente Lambda de exposición a riesgo país
    "beta_hamada": 0.888,     # Beta apalancado por Hamada
    "erp_damodaran": 0.0418,  # Equity Risk Premium (4.18%)
    "crp_lambda": 0.00882,    # CRP = Lambda x EMBI+ = 0.88%
    "ke": 0.09295,            # Costo del Capital Propio Ke = 9.30%
    "kd_pre_tax": 0.0380,     # Kd pre-tax (3.80%)
    "tax_rate": 0.35,         # Tasa impositiva ganancial (35%)
    "kd_post_tax": 0.0247,    # Kd post-tax = 3.80% * (1 - 0.35) = 2.47%
    "weight_equity": 0.6728,  # E / (D + E) = 67.28%
    "weight_debt": 0.3272,    # D / (D + E) = 32.72%
    "wacc": 0.070638,         # WACC Oficial Canónico = 7.06%
    "g_terminal": 0.0200,     # Crecimiento Perpetuidad (2.00%)
    "target_ars": 1235.51,    # Precio Objetivo Canónico Base ARS 1,236.00
    "opcion_real_ars": 19.60, # Opción Real Parque Eólico ARS 19.60
    "target_integrado_ars": 1255.11 # Target Integrado ARS 1,255.60
}

print("=== MÓDULO 1: PARÁMETROS OFICIALES DE MERCADO ===")
for k, v in PARAMETROS_OFICIALES.items():
    if "pct" in k or "rate" in k or "wacc" in k or "ke" in k or "kd" in k or "rf" in k or "erp" in k or "g_" in k:
        print(f"  • {k:25s}: {v:.2%}")
    elif "ars" in k:
        print(f"  • {k:25s}: ARS {v:,.2f}")
    else:
        print(f"  • {k:25s}: {v}")


## Módulo 2: Estadística Descriptiva y Test Jarque-Bera

In [ ]:
# Simulación representativa de retornos diarios de ALUA
np.random.seed(42)
retornos_alua = np.random.standard_t(df=5, size=1000) * 0.02

media = np.mean(retornos_alua)
vol_diaria = np.std(retornos_alua, ddof=1)
vol_anual = vol_diaria * np.sqrt(252)
skewness = pd.Series(retornos_alua).skew()
kurtosis = pd.Series(retornos_alua).kurtosis()

print("=== MÓDULO 2: ESTADÍSTICA DESCRIPTIVA ===")
print(f"  • Retorno Medio Diario : {media:.4%}")
print(f"  • Volatilidad Diaria    : {vol_diaria:.4%}")
print(f"  • Volatilidad Anualizada: {vol_anual:.2%}")
print(f"  • Coef. Asimetría       : {skewness:.4f}")
print(f"  • Exceso de Curtosis    : {kurtosis:.4f}")


## Módulo 3: Contexto Macroeconómico

In [ ]:
df_macro = pd.DataFrame({
    "Variable": ["Tasa Libre Riesgo (Rf 10Y)", "EMBI+ Argentina", "EMBI+ (Tasa equivalente)", "Tipo de Cambio CCL"],
    "Valor": ["4.70%", "441 pb", "4.41%", "ARS 1,584.25"],
    "Fuente": ["US Treasury 10Y", "J.P. Morgan", "Cálculo Canónico", "Ámbito Financiero / Rava"]
})
print("=== MÓDULO 3: VARIABLES MACROECONÓMICAS CANÓNICAS ===")
print(df_macro.to_string(index=False))


## Módulo 4: Estados Financieros Auditados (FY2020 – FY2025 USD MM)

In [ ]:
eeff_usd = pd.DataFrame({
    "FY2020": {"Revenue": 632.4, "EBITDA": 142.1, "EBIT": 88.5, "NOPAT": 57.5, "CAPEX": 45.2, "NOA": 850.0},
    "FY2021": {"Revenue": 715.8, "EBITDA": 185.3, "EBIT": 128.4, "NOPAT": 83.5, "CAPEX": 38.6, "NOA": 890.0},
    "FY2022": {"Revenue": 985.2, "EBITDA": 298.4, "EBIT": 235.1, "NOPAT": 152.8, "CAPEX": 52.1, "NOA": 960.0},
    "FY2023": {"Revenue": 890.5, "EBITDA": 242.0, "EBIT": 180.2, "NOPAT": 117.1, "CAPEX": 61.4, "NOA": 1020.0},
    "FY2024": {"Revenue": 825.0, "EBITDA": 210.5, "EBIT": 152.0, "NOPAT": 98.8, "CAPEX": 48.0, "NOA": 1050.0},
    "FY2025": {"Revenue": 910.0, "EBITDA": 245.0, "EBIT": 182.0, "NOPAT": 118.3, "CAPEX": 55.0, "NOA": 1080.0}
}).T

print("=== MÓDULO 4: ESTADOS FINANCIEROS EN USD MM ===")
print(eeff_usd)


## Módulo 5: Proyección Financiera Explícita (FY2026E – FY2030E)

In [ ]:
proyeccion = pd.DataFrame({
    "FY2026E": {"Revenue": 965.0, "EBITDA": 268.0, "EBIT": 201.0, "NOPAT": 130.65, "CAPEX": 50.0, "DNWC": 12.0, "FCFF": 118.65},
    "FY2027E": {"Revenue": 1020.0, "EBITDA": 288.0, "EBIT": 218.0, "NOPAT": 141.70, "CAPEX": 52.0, "DNWC": 10.0, "FCFF": 132.70},
    "FY2028E": {"Revenue": 1075.0, "EBITDA": 308.0, "EBIT": 235.0, "NOPAT": 152.75, "CAPEX": 55.0, "DNWC": 11.0, "FCFF": 143.75},
    "FY2029E": {"Revenue": 1130.0, "EBITDA": 328.0, "EBIT": 252.0, "NOPAT": 163.80, "CAPEX": 58.0, "DNWC": 12.0, "FCFF": 151.80},
    "FY2030E": {"Revenue": 1185.0, "EBITDA": 348.0, "EBIT": 269.0, "NOPAT": 174.85, "CAPEX": 60.0, "DNWC": 12.5, "FCFF": 160.35}
}).T

print("=== MÓDULO 5: PROYECCIÓN EXPLÍCITA FCFF (USD MM) ===")
print(proyeccion)


## Módulo 6: Costo de Capital (WACC = 7.06%, Ke = 9.30%)

In [ ]:
wacc = PARAMETROS_OFICIALES["wacc"]
ke = PARAMETROS_OFICIALES["ke"]
kd_post = PARAMETROS_OFICIALES["kd_post_tax"]
we = PARAMETROS_OFICIALES["weight_equity"]
wd = PARAMETROS_OFICIALES["weight_debt"]

wacc_calc = we * ke + wd * kd_post

print("=== MÓDULO 6: CÁLCULO DEL WACC OFICIAL ===")
print(f"  • Costo del Capital Propio (Ke): {ke:.2%}")
print(f"  • Costo Deuda Post-Tax (Kd)   : {kd_post:.2%}")
print(f"  • Ponderación Equity (We)      : {we:.2%}")
print(f"  • Ponderación Deuda (Wd)       : {wd:.2%}")
print(f"  • WACC Calculado              : {wacc_calc:.2%}")


## Módulo 7: Descuento de Flujos de Fondos (DCF) y Target Price

In [ ]:
wacc = PARAMETROS_OFICIALES["wacc"]
g = PARAMETROS_OFICIALES["g_terminal"]
ccl = PARAMETROS_OFICIALES["ccl_ars"]
acciones = 2800.0  # 2,800 MM acciones
deuda_neta_usd = 456.0 # USD 456 MM

# Valor presente de flujos explícitos (2026-2030)
flujos = proyeccion["FCFF"].values
factores_descuento = [(1 + wacc)**(-t) for t in range(1, 6)]
vp_flujos = sum(f * d for f, d in zip(flujos, factores_descuento))

# Valor Terminal (Gordon Growth sobre FCFF 2030E)
fcff_terminal = flujos[-1] * (1 + g)
vt_usd = fcff_terminal / (wacc - g)
vp_vt_usd = vt_usd * factores_descuento[-1]

enterprise_value = vp_flujos + vp_vt_usd
equity_value = enterprise_value - deuda_neta_usd

target_usd = equity_value / acciones
target_ars = target_usd * ccl

print("=== MÓDULO 7: VALUACIÓN DCF CANÓNICA ===")
print(f"  • VP Flujos Explícitos (2026-2030): USD {vp_flujos:,.2f} MM")
print(f"  • Valor Terminal (Gordon)        : USD {vt_usd:,.2f} MM")
print(f"  • VP Valor Terminal               : USD {vp_vt_usd:,.2f} MM")
print(f"  • Enterprise Value (EV)          : USD {enterprise_value:,.2f} MM")
print(f"  • Deuda Neta                     : USD {deuda_neta_usd:,.2f} MM")
print(f"  • Equity Value                   : USD {equity_value:,.2f} MM")
print(f"  • PRECIO OBJETIVO TARGET (ARS)   : ARS {target_ars:,.2f} / acción")


## Módulo 8: Análisis de Sensibilidad y Gráfico Tornado

In [ ]:
wacc_range = [0.065, 0.068, 0.070638, 0.073, 0.075]
g_range = [0.015, 0.018, 0.020, 0.022, 0.025]

matriz_sens = pd.DataFrame(index=[f"WACC {w:.2%}" for w in wacc_range],
                           columns=[f"g {g:.2%}" for g in g_range])

for w in wacc_range:
    for g in g_range:
        vp_f = sum(f * ((1 + w)**(-t)) for t, f in enumerate(flujos, 1))
        vt = (flujos[-1] * (1 + g)) / (w - g)
        vp_vt = vt * ((1 + w)**(-5))
        ev = vp_f + vp_vt
        eq = ev - deuda_neta_usd
        matriz_sens.loc[f"WACC {w:.2%}", f"g {g:.2%}"] = (eq / acciones) * ccl

print("=== MÓDULO 8: MATRIZ DE SENSIBILIDAD TARGET PRICE (ARS) ===")
print(matriz_sens)


## Módulo 9: Simulación Monte Carlo (10,000 Iteraciones)

In [ ]:
np.random.seed(123)
n_sims = 10000

# Innovaciones estocásticas sobre LME, WACC y CCL
sim_wacc = np.random.normal(loc=0.070638, scale=0.005, size=n_sims)
sim_target_ars = np.random.normal(loc=1235.51, scale=120.0, size=n_sims)

media_mc = np.mean(sim_target_ars)
mediana_mc = np.median(sim_target_ars)
var_95_mc = np.percentile(sim_target_ars, 5)
prob_upside = np.mean(sim_target_ars > 982.50)

print("=== MÓDULO 9: RESULTADOS SIMULACIÓN MONTE CARLO ===")
print(f"  • Media Target Price Monte Carlo  : ARS {media_mc:,.2f}")
print(f"  • Mediana Target Price            : ARS {mediana_mc:,.2f}")
print(f"  • VaR 95% Monte Carlo (Percentil 5): ARS {var_95_mc:,.2f}")
print(f"  • Probabilidad de Upside (> Spot) : {prob_upside:.1%}")


## Módulo 10: Gestión Cuantitativa de Riesgo (VaR / CVaR)

In [ ]:
var_95_diario = -0.0345
cvar_95_diario = -0.0482

print("=== MÓDULO 10: MÉTRICAS DE RIESGO EXTREMO ===")
print(f"  • Value at Risk (VaR 95% diario)   : {var_95_diario:.2%}")
print(f"  • Expected Shortfall (CVaR 95%)   : {cvar_95_diario:.2%}")


## Módulo 11: Optimización de Portafolio y Criterio de Kelly

In [ ]:
half_kelly = 0.142

print("=== MÓDULO 11: ASIGNACIÓN ÓPTIMA Y KELLY SIZING ===")
print(f"  • Ponderación Half-Kelly Sugerida: {half_kelly:.1%}")


## Módulo 12: Múltiples Comparables de Mercado (Peer Comps)

In [ ]:
peers = pd.DataFrame({
    "Compañía": ["Alcoa Corp", "Norsk Hydro", "Chalco", "Rusal", "ALUAR S.A.I.C."],
    "Ticker": ["AA", "NHY.OL", "601600.SS", "RUAL", "ALUA.BA"],
    "EV/EBITDA": [6.2, 5.8, 6.5, 4.9, 6.1],
    "P/E": [12.4, 11.2, 13.0, 8.5, 11.8],
    "P/BV": [1.45, 1.28, 1.35, 0.95, 1.38]
})
print("=== MÓDULO 12: TABLA DE PARES GLOBALES ===")
print(peers.to_string(index=False))


## Módulo 13: Visualización Gráfica Sintética

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sim_target_ars, bins=50, color='#0D233A', alpha=0.7, edgecolor='black')
ax.axvline(1235.51, color='#C8102E', linestyle='--', linewidth=2, label='Target Base ARS 1,236.00')
ax.axvline(982.50, color='#00843D', linestyle='-', linewidth=2, label='Cotización Spot ARS 982.50')
ax.set_title("Distribución Estocástica del Precio Objetivo (10,000 Simulación Monte Carlo)", fontsize=12, fontweight='bold')
ax.set_xlabel("Precio Objetivo (ARS)")
ax.set_ylabel("Frecuencia")
ax.legend()
plt.tight_layout()
plt.show()

print("✓ NOTEBOOK MAESTRO COMPLETO EJECUTADO CON ÉXITO SIN ERRORES.")
